In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df
import pickle
import cudf

Enabled rmm statistics


In [2]:
%%RecordEvent
# %load_ext cudf.pandas
# import dias.rewriter
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import os
# STEFANOS: Conditionally import Modin Pandas
if "IREWR_WITH_MODIN" in os.environ and os.environ["IREWR_WITH_MODIN"] == "True":
    # STEFANOS: Import Modin Pandas
    import os
    os.environ["MODIN_ENGINE"] = "ray"
    import ray
    ray.init(num_cpus=int(os.environ['MODIN_CPUS']), runtime_env={'env_vars': {'__MODIN_AUTOIMPORT_PANDAS__': '1'}})
    import modin.pandas as pd
else:
    # STEFANOS: Import regular Pandas
    import pandas as pd
import seaborn as sns  # data visualization
import matplotlib.pyplot as plt


# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/acnh-villager-popularity/acnh_villager_data.csv
/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/animal-crossing-new-horizons-nookplaza-dataset/villagers.csv


# Introduction
The goal of this project is to analyse the relationship between animal crossing new horizon villager popularity amongst the player base and certain villager attributes. 

We will be analysing the Gender,Personality, Species, and Style of a villager. 

# Data Initilization and Cleaning

In [3]:
import time
start_time = time.perf_counter()

In [4]:
### cell 0 ###

vlgr_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/animal-crossing-new-horizons-nookplaza-dataset/villagers.csv")
popul_df = pd.read_csv("/home/dias-benchmarks/notebooks/ampiiere/animal-crossing-villager-popularity-analysis/input/acnh-villager-popularity/acnh_villager_data.csv")

# -- STEFANOS -- Replicate Data

In [5]:
### cell 1 ###

factor = 100
popul_df = pd.concat([popul_df]*factor, ignore_index=True)
vlgr_df = pd.concat([vlgr_df]*factor, ignore_index=True)
print(popul_df.info())
vlgr_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41300 entries, 0 to 41299
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tier    41300 non-null  int64 
 1   rank    41300 non-null  int64 
 2   name    41300 non-null  object
dtypes: int64(2), object(1)
memory usage: 968.1+ KB
None
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39100 entries, 0 to 39099
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Name             39100 non-null  object
 1   Species          39100 non-null  object
 2   Gender           39100 non-null  object
 3   Personality      39100 non-null  object
 4   Hobby            39100 non-null  object
 5   Birthday         39100 non-null  object
 6   Catchphrase      39100 non-null  object
 7   Favorite Song    39100 non-null  object
 8   Style 1          39100 non-null  object
 9   Style 2          39100 non-null  object
 10  C

In [6]:
### cell 2 ###

vlgr_df.head()

,Name,Species,Gender,Personality,Hobby,Birthday,Catchphrase,Favorite Song,Style 1,Style 2,Color 1,Color 2,Wallpaper,Flooring,Furniture List,Filename,Unique Entry ID
0,Admiral,Bird,Male,Cranky,Nature,27-Jan,aye aye,Steep Hill,Cool,Cool,Black,Blue,dirt-clod wall,tatami,717;1849;7047;2736;787;5970;3449;3622;3802;410...,brd06,B3RyfNEqwGmcccRC3
1,Agent S,Squirrel,Female,Peppy,Fitness,2-Jul,sidekick,Go K.K. Rider,Active,Simple,Blue,Black,concrete wall,colorful tile flooring,7845;7150;3468;4080;290;3971;3449;1708;4756;25...,squ05,SGMdki6dzpDZyXAw5
2,Agnes,Pig,Female,Big Sister,Play,21-Apr,snuffle,K.K. House,Simple,Elegant,Pink,White,gray molded-panel wall,arabesque flooring,4129;7236;7235;7802;896;3428;4027;7325;3958;71...,pig17,jzWCiDPm9MqtCfecP
3,Al,Gorilla,Male,Lazy,Fitness,18-Oct,ayyyeee,Go K.K. Rider,Active,Active,Red,White,concrete wall,green rubber flooring,1452;4078;4013;833;4116;3697;7845;3307;3946;39...,gor08,LBifxETQJGEaLhBjC
4,Alfonso,Alligator,Male,Lazy,Play,9-Jun,it'sa me,Forest Life,Simple,Simple,Red,Blue,yellow playroom wall,green honeycomb tile,4763;3205;3701;1557;3623;85;3208;3584;4761;121...,crd00,REpd8KxB8p9aGBRSE


In [7]:
### cell 3 ###

popul_df.head()

,tier,rank,name
0,1,1,Raymond
1,1,2,Marshal
2,1,3,Shino
3,1,4,Sherb
4,1,5,Sasha


### 1. Checking for null 

In [8]:
### cell 4 ###

vlgr_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39100 entries, 0 to 39099
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Name             39100 non-null  object
 1   Species          39100 non-null  object
 2   Gender           39100 non-null  object
 3   Personality      39100 non-null  object
 4   Hobby            39100 non-null  object
 5   Birthday         39100 non-null  object
 6   Catchphrase      39100 non-null  object
 7   Favorite Song    39100 non-null  object
 8   Style 1          39100 non-null  object
 9   Style 2          39100 non-null  object
 10  Color 1          39100 non-null  object
 11  Color 2          39100 non-null  object
 12  Wallpaper        39100 non-null  object
 13  Flooring         39100 non-null  object
 14  Furniture List   39100 non-null  object
 15  Filename         39100 non-null  object
 16  Unique Entry ID  39100 non-null  object
dtypes: object(17)
memory usage: 5.1

In [9]:
### cell 5 ###

popul_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41300 entries, 0 to 41299
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   tier    41300 non-null  int64 
 1   rank    41300 non-null  int64 
 2   name    41300 non-null  object
dtypes: int64(2), object(1)
memory usage: 968.1+ KB


### 2. Checking for mismatched names

In [10]:
### cell 6 ###

# There are some missing/non-matching names 
vlgr_df["Name"].isin(popul_df['name']).sum()

np.int64(38600)

In [11]:
### cell 7 ###

# Find names in popul_df that are not in vlgr_df
mismatch_names = popul_df[~popul_df["name"].isin(vlgr_df["Name"])]
mismatch_names

,tier,rank,name
2,1,3,Shino
4,1,5,Sasha
5,1,6,Ione
25,2,11,Cephalobot
57,3,18,Étoile
...,...,...,...
41187,6,51,Crackle(Spork)
41210,6,74,Ace
41215,6,79,Toby
41228,6,92,Frett


In [12]:
### cell 8 ###

# Optimized code

# Using a single replace call for multiple substitutions
popul_df['name'] = popul_df['name'].replace({
    'OHare': "O'Hare",
    'Buck(Brows)': 'Buck',
    'Renee': 'Renée',
    'WartJr': 'Wart Jr.',
    'Crackle(Spork)': 'Spork'
})

In [13]:
### cell 9 ###

# Checking if All names match
vlgr_df["Name"].isin(popul_df['name']).sum()

np.int64(39100)

In [14]:
### cell 10 ###

popul_df = popul_df[popul_df['name'].isin(vlgr_df['Name'])]

### 3. Merging the two Dataframes

In [15]:
### cell 11 ###

# Now that both df have same length, we can set index as names and combine the 2 dfs
popul_df.set_index('name', drop=True, inplace=True)
vlgr_df.set_index('Name', drop=True, inplace=True)

In [16]:
### cell 12 ###

combined_df = popul_df.merge(vlgr_df, left_index=True, right_index=True)

In [17]:
### cell 13 ###

# drop irrelevent columns
combined_df.drop(columns=['Furniture List', 'Filename', 'Unique Entry ID', "Wallpaper", "Flooring", "Birthday", "Favorite Song"], inplace=True)

#### Adding a new row named overall_ranking so we may know a villager's general ranking outside of their tier

In [18]:
### cell 14 ###

combined_df.sort_values(['tier', 'rank'], inplace=True)
combined_df['overall_ranking'] = np.arange(1, len(combined_df)+1)
combined_df.insert(2, 'overall_ranking', combined_df.pop('overall_ranking'))

#### Setting Baseline overall ranking mean to compare against

In [19]:
### cell 15 ###

overall_mean = combined_df.overall_ranking.mean()
print(f'The overall_mean is {overall_mean}, this would serve as a baseline for to compare against popularity performance of our features.')

The overall_mean is 1955000.5, this would serve as a baseline for to compare against popularity performance of our features.


In [20]:
### cell 16 ###

combined_df.columns

Index(['tier', 'rank', 'overall_ranking', 'Species', 'Gender', 'Personality',
       'Hobby', 'Catchphrase', 'Style 1', 'Style 2', 'Color 1', 'Color 2'],
      dtype='object')

# Exploratory Data Analysis
As a preface, a higher overall_ranking would mean performing worse on the popularity rankings.
### 1. Gender

In [21]:
### cell 17 ###

combined_df['Gender'].value_counts()

Gender
Male      2040000
Female    1870000
Name: count, dtype: int64

In [22]:
### cell 18 ###

combined_df.groupby(['tier', 'Gender']).size()

tier  Gender
1     Female     60000
      Male       60000
2     Female    140000
      Male      100000
3     Female    180000
      Male      110000
4     Female    300000
      Male      250000
5     Female    580000
      Male      550000
6     Female    610000
      Male      970000
dtype: int64

For gender, there seems to be a disproporationate amount of male villagers in the lowest tier(6th tier) than female villagers, compared to other tiers. Discounting Tier 6, The number of male and female villagers are fairly even, with Male villagers having a slight lead in all tiers(except tier 6).

In [23]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(5, 5))
# plt.axhline(overall_mean, color='r')
# sns.boxplot(x="Gender", y='overall_ranking', data=combined_df)

Female villagers generally perform better than Male villagers in terms of overall ranking. 

In [24]:
### cell 19 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Gender", aggfunc='count')

Gender,Female,Male
tier,,
1,60000,60000
2,140000,100000
3,180000,110000
4,300000,250000
5,580000,550000
6,610000,970000


### 2. Species

In [25]:
### cell 20 ###

### Optimized Code ###
species_ranking = combined_df.groupby('Species', as_index=False)['overall_ranking'].mean(numeric_only=True).sort_values('overall_ranking')
species_ranking

,Species,overall_ranking
25,Octopus,1.683338e+05
9,Deer,5.180005e+05
34,Wolf,7.213641e+05
5,Cat,9.402179e+05
21,Koala,1.336112e+06
10,Dog,1.392500e+06
8,Cub,1.426876e+06
17,Hamster,1.467500e+06
32,Squirrel,1.481112e+06
30,Rhino,1.548334e+06


In [26]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(30,5))
# sns.set(font_scale=1.4)
# plt.xticks(rotation=45)
# plt.axhline(overall_mean, color='r')
# sns.scatterplot(x='Species', y="overall_ranking", data=species_ranking,label='mean overall-ranking', s=300)

Octopus, deer, wolves, cats and Koalas are most likely to be popular; while Kangaroos, Hippos, Mouse Pigs and Gorillas are the least likely to be popular. 

In [27]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(30, 10))
# plt.axhline(overall_mean, color='r')
# sns.scatterplot(x="Species", y='overall_ranking', hue='tier', s=100, data=combined_df)

Although Octopuses seem to be ranking highly in part due to the low amount of Octopuses amongst the villagers. 
Interesting trend can be observed, there exists a ranking cap for low ranking speices, for example, none of the Gorilla villagers have a ranking lower than 200, it is heavily skewed, and not normally distributed.  Indicating a clear non-preference for certain species by the playerbase. 

### 3. Personality

In [28]:
### cell 21 ###

combined_df.Personality.value_counts()

Personality
Lazy          600000
Normal        590000
Snooty        550000
Jock          550000
Cranky        550000
Peppy         490000
Smug          340000
Big Sister    240000
Name: count, dtype: int64

In [29]:
### cell 22 ###

personality_ranking = (combined_df
                     .groupby('Personality', as_index=False)['overall_ranking']
                     .mean(numeric_only=True)
                     .sort_values('overall_ranking'))

In [30]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(20,5))
# sns.set(font_scale=1.4)
# plt.xticks(rotation=45)
# plt.axhline(overall_mean, color='r')
# sns.scatterplot(x='Personality', y="overall_ranking", data=personality_ranking,label='mean personality ranking', s=300)

The playerbase seems to have a preference for Big sister, Normal, Peppy and sometimes Lazy type villagers.
While they dislike Cranky, Jock and Snooty villagers. 

In [31]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(10, 10))
# plt.axhline(overall_mean, color='r')
# sns.boxplot(x="Personality", y='overall_ranking', data=combined_df)

There seems to be a clear preference for Big Sister, Peppy and Normal Personality villagers, they have means below overall mean. Rankings are fairly normally distributed except for Smug villagers. On the other hand, Cranky and Snooty both have a mean clearly above the overall mean.

In [32]:
### cell 23 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Personality", aggfunc='count')

Personality,Big Sister,Cranky,Jock,Lazy,Normal,Peppy,Smug,Snooty
tier,,,,,,,,
1,NaN,NaN,NaN,40000.0,40000.0,NaN,20000.0,20000.0
2,10000.0,20000.0,20000.0,40000.0,60000.0,60000.0,20000.0,10000.0
3,50000.0,40000.0,30000.0,20000.0,80000.0,40000.0,20000.0,10000.0
4,60000.0,50000.0,50000.0,90000.0,80000.0,100000.0,60000.0,60000.0
5,80000.0,130000.0,180000.0,190000.0,190000.0,160000.0,50000.0,150000.0
6,40000.0,310000.0,270000.0,220000.0,140000.0,130000.0,170000.0,300000.0


### 4. Style

In [33]:
### cell 24 ###

def compute_style_ranking(combined_df, style_column):
    return (combined_df.groupby(style_column, as_index=False)['overall_ranking']
                     .mean(numeric_only=True)
                     .sort_values('overall_ranking'))

style_ranking1 = compute_style_ranking(combined_df, 'Style 1')
style_ranking2 = compute_style_ranking(combined_df, 'Style 2')

In [34]:
### cell 25 ###

style_ranking = style_ranking1.copy()
style_ranking["overall_ranking"] = (style_ranking1["overall_ranking"] + style_ranking2["overall_ranking"]) / 2

In [35]:
### cell 26 ###

style_ranking

,Style 1,overall_ranking
2,Cute,1.375134e+06
5,Simple,1.860020e+06
3,Elegant,2.094237e+06
1,Cool,2.162892e+06
4,Gorgeous,2.190078e+06
0,Active,2.256700e+06


In [36]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(20,5))
# sns.set(font_scale=1.4)
# plt.xticks(rotation=45)
# plt.axhline(overall_mean, color='r')
# sns.scatterplot(x='Style 1', y="overall_ranking", data=style_ranking, s=300)

A very clear preference for Cute styled villagers. Simple Styled Villagers have a ranking mean just about equal to the overall mean, while other style villagers have a slightly above overall mean mean. 

In [37]:
## -- STEFANOS: Disable plotting
# plt.figure(figsize=(7, 7))
# plt.axhline(overall_mean, color='r')
# sns.boxplot(x="Style 1", y='overall_ranking', data=combined_df)
# plt.title('Style 1')
# plt.figure(figsize=(7, 7))
# plt.axhline(overall_mean, color='r')
# sns.boxplot(x="Style 2", y='overall_ranking', data=combined_df)
# plt.title('Style 2')

The clear preference is Cute style dressing, in both Style columns. In particular, in Style 2 column Cute Styled Villagers have a higher concetration in lower rankings. Other styles seem to have a fairly normally distributed ranking, with the exception of Active Style Villagers in Style 1, right skewed, but the ranking mean is significantly above the overall ranking mean.  

In [38]:
### cell 27 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Style 1", aggfunc='count')

Style 1,Active,Cool,Cute,Elegant,Gorgeous,Simple
tier,,,,,,
1,NaN,NaN,30000.0,20000.0,10000.0,60000.0
2,20000.0,20000.0,80000.0,10000.0,20000.0,90000.0
3,30000.0,70000.0,80000.0,50000.0,NaN,60000.0
4,20000.0,110000.0,110000.0,70000.0,60000.0,180000.0
5,150000.0,170000.0,140000.0,170000.0,110000.0,390000.0
6,280000.0,310000.0,190000.0,220000.0,180000.0,400000.0


In [39]:
### cell 28 ###

pd.pivot_table(combined_df, index = 'tier', values = 'Catchphrase', columns="Style 2", aggfunc='count')

Style 2,Active,Cool,Cute,Elegant,Gorgeous,Simple
tier,,,,,,
1,NaN,20000.0,70000.0,10000.0,NaN,20000.0
2,20000.0,20000.0,60000.0,20000.0,10000.0,110000.0
3,20000.0,30000.0,100000.0,30000.0,20000.0,90000.0
4,80000.0,70000.0,130000.0,80000.0,80000.0,110000.0
5,130000.0,150000.0,180000.0,150000.0,180000.0,340000.0
6,250000.0,300000.0,80000.0,270000.0,250000.0,430000.0


In [40]:
end_time = time.perf_counter()
print(end_time-start_time)

8.566506107999885


# Conclusion
We may come to the conclusion, that the following attributes contribute to a villager's popularity:
- Gender: Despite Female Villagers having in general better popularity, this is likely due to the overwheling prescence of male villagers in the lowest tier. Other than the lowest tier, Male villagers in general perform slightly better.
- Species: Octopus, Wolf, Deer and Cat villagers perform the best. 
- Personality: Big Sister, Normal and Peppy villagers are in general the most popular. 
- Style: Cute Style villagers are very clearly the most popular